# 2025 Race Data Retrieval Pipeline (FastF1)
This notebook retrieves completed **Race** sessions for the 2025 season and exports a clean strategy dataset.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import fastf1

BASE_DIR = Path('..').resolve()
OUT_DIR = BASE_DIR / 'data'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = BASE_DIR / '.fastf1_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))


In [ ]:
def estimate_fuel_load(lap_number: int, total_laps: int, start_fuel_kg: float = 100.0, burn_per_lap: float = 1.7) -> float:
    return max(0.0, start_fuel_kg - (lap_number - 1) * burn_per_lap)

def safe_session_load(year: int, event_name: str):
    try:
        sess = fastf1.get_session(year, event_name, 'R')
        sess.load(laps=True, telemetry=False, weather=True)
        return sess
    except Exception as exc:
        if 'RateLimitExceeded' in str(exc) or '429' in str(exc):
            print(f'[WARN] Rate limit for {event_name}: {exc}')
            return None
        print(f'[WARN] Failed {event_name}: {exc}')
        return None


In [ ]:
schedule = fastf1.get_event_schedule(2025, include_testing=False)
schedule = schedule[schedule['Session5'].notna()].copy()
# Session5 generally corresponds to the Race session in FastF1 schedule tables
completed = schedule[schedule['EventDate'] <= pd.Timestamp.utcnow()]
print('Completed 2025 events:', len(completed))
completed[['EventName','EventDate']].head()


In [ ]:
all_rows = []

for _, event in completed.iterrows():
    event_name = event['EventName']
    sess = safe_session_load(2025, event_name)
    if sess is None:
        continue

    weather = sess.weather_data if hasattr(sess, 'weather_data') else pd.DataFrame()
    air_temp = float(weather['AirTemp'].mean()) if 'AirTemp' in weather else np.nan
    track_temp = float(weather['TrackTemp'].mean()) if 'TrackTemp' in weather else np.nan

    laps = sess.laps.copy()
    laps = laps[laps['LapTime'].notna()]
    if laps.empty:
        continue

    total_laps = int(laps['LapNumber'].max())

    for _, lap in laps.iterrows():
        lap_no = int(lap['LapNumber'])
        all_rows.append({
            'Season': 2025,
            'EventName': event_name,
            'Driver': lap.get('Driver'),
            'LapNumber': lap_no,
            'TyreLife': lap.get('TyreLife'),
            'Compound': str(lap.get('Compound') or 'UNKNOWN').upper(),
            'TrackStatus': lap.get('TrackStatus'),
            'LapTimeSeconds': float(lap['LapTime'].total_seconds()),
            'FuelLoadKgEst': estimate_fuel_load(lap_no, total_laps),
            'AirTemp': air_temp,
            'TrackTemp': track_temp,
        })

df = pd.DataFrame(all_rows)
print('Rows collected:', len(df))
df.head()


In [ ]:
# Strict race-only cleanup + strategy-focused feature prep
df = df.dropna(subset=['TyreLife','Compound','LapTimeSeconds','TrackTemp'])
df['TyreLife'] = pd.to_numeric(df['TyreLife'], errors='coerce')
df = df.dropna(subset=['TyreLife'])

df['ThermalDegIndex'] = df['TyreLife'] * (df['TrackTemp'] / 35.0)
df['MechanicalWearIndex'] = df['TyreLife'] * (1.0 + df['LapNumber'] / df['LapNumber'].max())

out_path = OUT_DIR / 'strategy_2025_race_only.csv'
df.to_csv(out_path, index=False)
print('Saved:', out_path)
print(df[['Compound','TyreLife','TrackTemp','LapTimeSeconds']].describe(include='all'))
